<a href="https://colab.research.google.com/github/deepsharma26/SIRT1/blob/Descriptor_gen/RDkit_discriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit-pypi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 14.4 MB/s eta 0:00:00


In [2]:
!pip install mordred

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.8/128.8 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.7 MB/s eta 0:00:00
  Created wheel for mordred: filename=mordred-1.2.0-py3-none-any.whl size=176718 sha256=4c3bf4749279f3e7af0fcdb24255e7ee13dc7486e6d9098443c3eaebd0a257a8
  Stored in directory: /root/.cache/pip/wheels/8b/30/0b/84e3f6775306e74cf5957ee4d16b10bf3927dcec44cc23d5f2
Successfully built mordred
  Attempting uninstall: networkx
    Found existing installation: networkx 3.4.2
    Uninstalling networkx-3.4.2:
      Successfully uninstalled networkx-3.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatib

In [3]:
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [4]:
dataset=pd.read_csv('/content/SIRT1_05_bioactivity_data_2class_pIC50.csv')

In [5]:
df=dataset.dropna(subset=['canonical_smiles'])

In [6]:
df

,Unnamed: 0,molecule_chembl_id,canonical_smiles,class,MW,LogP,NumHDonors,NumHAcceptors,pIC50
0,0,CHEMBL420311,NC(=O)C1CCCc2c1[nH]c1ccc(Cl)cc21,active,248.713,2.72650,2.0,1.0,7.008774
1,1,CHEMBL115600,Cc1ccc2[nH]c3c(c2c1)CCCC3C(N)=O,active,228.295,2.38152,2.0,1.0,6.688246
2,3,CHEMBL446446,NC(=O)C1CCCCc2c1[nH]c1ccc(Cl)cc21,active,262.740,3.11660,2.0,1.0,6.906578
3,4,CHEMBL171137,CCOC(=O)C1CCCc2c1[nH]c1ccc(Cl)cc21,inactive,277.751,3.80430,1.0,2.0,4.000000
4,5,CHEMBL171955,O=C(O)C1CCCc2c1[nH]c1ccc(Cl)cc21,inactive,249.697,3.32580,2.0,1.0,4.000000
...,...,...,...,...,...,...,...,...,...
724,924,CHEMBL2093745,O=C(NCCc1c[nH]c2ccccc12)c1n[nH]c2ccccc12,inactive,304.353,3.01670,3.0,2.0,5.000000
725,927,CHEMBL5405254,COc1ccc(N2CCN(c3nc(-c4ccccc4)nc4ccccc34)CC2)cc1,inactive,396.494,4.63200,0.0,5.0,4.397940
726,930,CHEMBL5416344,O=C(O)CCNC(=S)NCCCNc1nc(Nc2ccccc2Cl)ncc1C(=O)N...,inactive,582.130,4.24410,5.0,7.0,3.948462
727,931,CHEMBL5424599,O=C(O)CCNC(=S)NCCCNc1nc(Nc2ccccc2)nc(Nc2ccccc2...,inactive,501.016,3.75300,6.0,8.0,4.659556


In [7]:
def RDkit_descriptors(smiles):
    mols = [Chem.MolFromSmiles(i) for i in smiles]
    calc = MoleculeDescriptors.MolecularDescriptorCalculator([x[0] for x in Descriptors._descList])
    desc_names = calc.GetDescriptorNames()

    Mol_descriptors = []
    for mol in mols:
        # add hydrogens to molecules
        mol = Chem.AddHs(mol)
        # Calculate all 200 descriptors for each molecule
        descriptors = calc.CalcDescriptors(mol)
        Mol_descriptors.append(descriptors)
    return Mol_descriptors, desc_names

# Split the SMILES into chunks of 100,000 for faster processing
chunk_size = 100000
chunks = [df[i:i+chunk_size] for i in range(0, len(df), chunk_size)]

total_chunks=len(chunks)
total_time=0

# Check if there is an existing output file
if os.path.isfile('RDkit_descriptors.csv'):
    existing_data = pd.read_csv('RDkit_descriptors.csv', index_col=0)
else:
    existing_data = pd.DataFrame()
    # Calculate descriptors for each chunk and concatenate the results
for i, chunk in enumerate(tqdm(chunks, desc='Processing', total=len(chunks))):
    # Check if this chunk has already been processed
    if len(existing_data) >= len(chunk):
        continue
    # Calculate descriptors for this chunk
    descriptors, desc_names = RDkit_descriptors(chunk['canonical_smiles'])
    # Convert the descriptors to a dataframe
    df_with_200_descriptors = pd.DataFrame(descriptors, columns=desc_names,)
    # Add the chunk index as a new column
    df_with_200_descriptors['chunk_index'] = i
    # Append the data to the existing data
    existing_data = pd.concat([existing_data, df_with_200_descriptors], axis=0)
    # Save the data after each chunk
    existing_data.to_csv('RDkit_descriptors.csv')

# Save the final data
existing_data.to_csv('RDkit_descriptors.csv')



Processing: 100%|██████████| 1/1 [00:41<00:00, 41.88s/it]
